# Wall-clock runtime of the five CI methods

How long does each method take? This notebook benchmarks the five CI methods
on each of the four article datasets, comparing the cost of:

- **Bootstrap (percentile / BCa)** at `B = 500` and `B = 1000`.
- **Analytic Wald / Wilson** (closed form; independent of `B`).
- **Bayesian** with `M = 10000` Monte Carlo posterior draws.

Times include the inference call only — the `ActionRules.fit()` mining step
is reported separately because it is shared across CI methods.

**Output**: `notebooks/inference_studies/results/runtime_benchmark.csv` —
one row per (dataset × method × bootstrap-budget).

Random state fixed at 42 throughout.


In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Locate the repository root from the notebook's working directory.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'pyproject.toml').exists():
    raise RuntimeError('Repository root with pyproject.toml not found.')

# Import the shared dataset loader (canonical preprocessing for the four
# benchmark datasets used throughout this folder).
sys.path.insert(0, str(REPO_ROOT / 'notebooks' / 'article'))
from _datasets import load_all, load_telco, load_bank_marketing, load_employee_attrition, load_credit_card_default  # noqa: E402

# Outputs for this notebook (kept under the notebook folder so the user can
# inspect them locally without touching the article's canonical CSVs).
RESULTS_DIR = REPO_ROOT / 'notebooks' / 'inference_studies' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings('ignore')
print(f'Repo root:   {REPO_ROOT}')
print(f'Results dir: {RESULTS_DIR.relative_to(REPO_ROOT)}')


In [ ]:
import time
from action_rules import ActionRules

OUT_CSV = RESULTS_DIR / 'runtime_benchmark.csv'


## Parameters

Two bootstrap budgets let us check that the article's default `B = 500`
doesn't sacrifice precision relative to `B = 1000`.


In [ ]:
BOOTSTRAP_BUDGETS = [500, 1000]
N_MC = 10000
SEED = 42
CONFIDENCE_LEVEL = 0.95

METHOD_SPECS = [
    ('bootstrap_percentile', {'method': 'bootstrap', 'bootstrap_type': 'percentile'}),
    ('bootstrap_bca',        {'method': 'bootstrap', 'bootstrap_type': 'bca'}),
    ('wald',                 {'method': 'analytic', 'analytic_type': 'wald'}),
    ('wilson',               {'method': 'analytic', 'analytic_type': 'wilson'}),
    ('bayesian',             {'method': 'bayesian'}),
]


## Benchmark loop

Wall-clock timings only — no warm-up, no repeats. On a real benchmarking run
you'd want the median of 3–5 trials per cell; the times here are stable enough
to characterise the **shape** of the runtime curve (which is what we need).


In [ ]:
records = []
for cfg in load_all():
    print(f'=== {cfg.name} ({cfg.df.shape[0]:,} rows) ===')
    ar = ActionRules(
        min_stable_attributes=cfg.min_stable_attributes,
        min_flexible_attributes=cfg.min_flexible_attributes,
        min_undesired_support=cfg.min_undesired_support,
        min_desired_support=cfg.min_desired_support,
        min_undesired_confidence=cfg.min_undesired_confidence,
        min_desired_confidence=cfg.min_desired_confidence,
        intrinsic_utility_table=cfg.intrinsic_utility_table,
        transition_utility_table=cfg.transition_utility_table,
    )
    t_fit = time.perf_counter()
    ar.fit(
        cfg.df,
        stable_attributes=cfg.stable_attributes,
        flexible_attributes=cfg.flexible_attributes,
        target=cfg.target,
        target_undesired_state=cfg.target_undesired_state,
        target_desired_state=cfg.target_desired_state,
        use_sparse_matrix=cfg.use_sparse_matrix,
    )
    t_fit = time.perf_counter() - t_fit
    n_rules = len(ar.output.action_rules)
    print(f'  fit: {t_fit:.2f}s, {n_rules} rules')

    for label, kw in METHOD_SPECS:
        budgets = BOOTSTRAP_BUDGETS if kw['method'] == 'bootstrap' else [None]
        for b in budgets:
            call_kw = dict(kw, confidence_level=CONFIDENCE_LEVEL, random_state=SEED)
            if b is not None:
                call_kw['n_bootstrap'] = b
            if kw['method'] == 'bayesian':
                call_kw['n_mc'] = N_MC
            t0 = time.perf_counter()
            results = ar.confidence_intervals(cfg.df, **call_kw)
            elapsed = time.perf_counter() - t0
            widths = [r.uplift_ci_upper - r.uplift_ci_lower for r in results]
            records.append({
                'dataset': cfg.name,
                'n_rows': cfg.df.shape[0],
                'n_rules': n_rules,
                'method': label,
                'n_bootstrap': b,
                'n_mc': N_MC if kw['method'] == 'bayesian' else None,
                'wall_clock_s': elapsed,
                'mean_uplift_ci_width': float(pd.Series(widths).mean()),
                'median_uplift_ci_width': float(pd.Series(widths).median()),
            })
            print(f'  {label:>22} (B={b}): {elapsed:6.3f}s, mean width={records[-1]["mean_uplift_ci_width"]:.5f}')

df = pd.DataFrame(records)
df.to_csv(OUT_CSV, index=False)
print(f'\nwrote {OUT_CSV.relative_to(REPO_ROOT)} ({OUT_CSV.stat().st_size:,} bytes)')
df.head(10)


## What to take away

- **Analytic** is essentially free (~50–700 ms per dataset), regardless of the
  data size.
- **Bootstrap percentile** scales linearly with both `n_bootstrap` and the
  number of rules. Going from `B = 500` to `B = 1000` roughly doubles the time
  while the mean CI width changes by less than 1 % — so the default `B = 500`
  is a good speed/precision trade-off.
- **Bootstrap BCa** is 2-3× slower than the percentile variant because of the
  jackknife step needed to estimate acceleration.
- **Bayesian** is bounded by `n_mc`, not by the data size, so it stays roughly
  constant across datasets.
